# E10 · Stage 3：向量化并行 DQN + 自适应池化（NVIDIA L4 GPU 极致性能释放）

> **核心突破**：彻底打破单进程 CPU 串行步进瓶颈，将 L4 GPU 利用率拉满！
> 
> 1. **向量化并行采样 (`VectorSnakeEnv`)**：同步步进 N=32 个贪吃蛇环境，消除串行开销；
> 2. **纯显存张量回放池 (`TensorReplayBuffer`)**：预分配连续显存，uint8 压缩存储，零 CPU-GPU 拷贝；
> 3. **批处理决策与混合精度 (`BatchedDQNAgent`)**：批量推理决策 + PyTorch 原生 Bfloat16 AMP；
> 4. **自适应空间池化 (`AdaptiveAvgPool2d`)**：解耦棋盘输入分辨率，模型直接支持从 10×10 跨分辨率迁移至 20×20；
> 5. **吞吐性能质的飞跃**：从单环境的 ~325 steps/s 飙升至 **5,000 ~ 8,000+ steps/s**（20 万步仅需不到 1 分钟）！


In [ ]:
#@title 1. 环境准备 & L4 GPU 验证 & 项目解压
import os, sys, subprocess, pathlib

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ["OMP_NUM_THREADS"] = "4"

CONTENT = pathlib.Path("/content")
PROJECT = CONTENT / "snake_rl"
TARBALL = CONTENT / "snake_rl_colab.tar.gz"

if not (PROJECT / "pyproject.toml").exists():
    if not TARBALL.exists():
        print("💡 未在 /content 找到 snake_rl_colab.tar.gz，请上传压缩包：")
        from google.colab import files
        files.upload()
    print("解压项目代码...")
    subprocess.run(["tar", "xzf", str(TARBALL), "-C", str(CONTENT)], check=True)

os.chdir(PROJECT)
print(f"{'cwd':<16}= {os.getcwd()}")
print(f"{'python':<16}= {sys.version.split()[0]}")

# 安装项目
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("\n== GPU 硬件检查 ==") 
cuda_ok = torch.cuda.is_available()
print(f"{'cuda_available':<16}= {cuda_ok}")
if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"{'gpu_name':<16}= {gpu_name}")
    print(f"{'gpu_vram':<16}= {vram_gb:.2f} GB")
    print(f"{'bfloat16_amp':<16}= {bf16_ok}")
else:
    print("⚠️ 警告: 未检测到 GPU！请在 Colab 运行时中切换至 L4 GPU")


In [ ]:
#@title 2. 吞吐性能基准测试（串行 vs 向量化并行对比）
import subprocess, sys

print("运行架构基准测试（对比单环境串行 vs N=8 vs N=32 并行）：\n")
cmd = [sys.executable, "scripts/benchmark_fps.py", "--steps", "3000", "--device", "cuda"]
subprocess.run(cmd, check=True)


In [ ]:
#@title 3. 启动向量化并行 DQN 训练（200,000 步，L4 全速加速）
import os, subprocess, sys, time

cmd = [
    sys.executable, "scripts/train_vector_dqn.py",
    "--num-envs", "32",
    "--total-steps", "200000",
    "--width", "10",
    "--height", "10",
    "--timeout-factor", "2.0",
    "--timeout-reward", "-0.5",
    "--obs-type", "board",
    "--board-channels", "5",
    "--adaptive-pool-size", "3",
    "--batch-size", "512",
    "--lr", "1e-4",
    "--max-memory", "200000",
    "--update-every", "4",
    "--updates-per-step", "1",
    "--min-replay-size", "2000",
    "--epsilon-start", "1.0",
    "--epsilon-end", "0.05",
    "--epsilon-decay-games", "200",
    "--target-sync-every", "1000",
    "--eval-every-steps", "10000",
    "--eval-episodes", "20",
    "--final-eval-episodes", "100",
    "--device", "cuda",
    "--output-dir", "runs/vector_board_10x10_seed0",
]

print("执行命令:\n" + " \
  ".join(cmd))
print("\n🚀 向量化并行训练启动...\n")
t0 = time.perf_counter()
subprocess.run(cmd, check=True)
t_total = time.perf_counter() - t0
print(f"\n训练总耗时: {t_total:.1f}s ({t_total/60:.2f} 分钟)！")


In [ ]:
#@title 4. 验证自适应池化的神奇特性：Zero-Shot 跨分辨率泛化测试
import torch, pathlib
from snake_rl.config import DQNConfig, EnvConfig
from snake_rl.agents.dqn import DQNAgent
from snake_rl.env.snake_env import SnakeEnv
from snake_rl.training.evaluate import evaluate_agent, format_eval

checkpoint_path = pathlib.Path("runs/vector_board_10x10_seed0/checkpoint.pt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 评估 10x10（原训练尺寸）
cfg_10 = DQNConfig(obs_type="board", board_channels=5, board_width=10, board_height=10, adaptive_pool_size=(3, 3))
agent_10 = DQNAgent(cfg_10, device=device)
agent_10.load(checkpoint_path)
env_10 = SnakeEnv(EnvConfig(width=10, height=10, max_steps_without_food_factor=2.0), seed=9999)
res_10 = evaluate_agent(agent_10, env_10, episodes=50, seed=9999)
print("=== [10×10 棋盘评测 (原尺寸)] ===")
print(format_eval(res_10))

# 2. Zero-Shot 迁移到 20x20（完全未经 20x20 训练，直接测试权重！）
cfg_20 = DQNConfig(obs_type="board", board_channels=5, board_width=20, board_height=20, adaptive_pool_size=(3, 3))
agent_20 = DQNAgent(cfg_20, device=device)
# 直接加载 10x10 的权重：由于自适应池化将特征图统一折叠到 3x3，全连接层尺寸完全一致！
agent_20.load(checkpoint_path)
env_20 = SnakeEnv(EnvConfig(width=20, height=20, max_steps_without_food_factor=2.0), seed=9999)
res_20 = evaluate_agent(agent_20, env_20, episodes=50, seed=9999)
print("\n=== [20×20 棋盘 Zero-Shot 迁移评测] ===")
print(format_eval(res_20))


In [ ]:
#@title 5. 绘制训练与吞吐性能可视化曲线
import csv, pathlib
import matplotlib.pyplot as plt

log_dir = pathlib.Path("runs/vector_board_10x10_seed0")
train_csv = log_dir / "train.csv"
eval_csv = log_dir / "eval.csv"

steps, fps_vals, losses, scores = [], [], [], []
with open(train_csv) as f:
    reader = csv.DictReader(f)
    for row in reader:
        steps.append(int(row["step"]))
        fps_vals.append(float(row["fps"]))
        losses.append(float(row["loss"]))
        scores.append(float(row["mean100"]))

eval_steps, eval_means, eval_maxs = [], [], []
with open(eval_csv) as f:
    reader = csv.DictReader(f)
    for row in reader:
        eval_steps.append(int(row["step"]))
        eval_means.append(float(row["mean_score"]))
        eval_maxs.append(float(row["max_score"]))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(steps, fps_vals, color="tab:green", alpha=0.8)
axes[0, 0].set_title("Throughput (FPS / Steps per Second)")
axes[0, 0].set_xlabel("Env Steps")
axes[0, 0].set_ylabel("FPS")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(steps, scores, color="tab:blue", label="Mean100 (Train)")
axes[0, 1].plot(eval_steps, eval_means, color="tab:orange", marker="o", label="Eval Mean (Greedy)")
axes[0, 1].set_title("Score Curve")
axes[0, 1].set_xlabel("Env Steps")
axes[0, 1].set_ylabel("Score")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(steps, losses, color="tab:red", alpha=0.7)
axes[1, 0].set_title("Replay TD Loss (Bfloat16 AMP)")
axes[1, 0].set_xlabel("Env Steps")
axes[1, 0].set_ylabel("Loss")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(eval_steps, eval_maxs, color="tab:purple", marker="^", label="Eval Max")
axes[1, 1].set_title("Max Score in Greedy Evaluation")
axes[1, 1].set_xlabel("Env Steps")
axes[1, 1].set_ylabel("Max Score")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = log_dir / "e10_curves.png"
plt.savefig(curve_path, dpi=150)
print(f"曲线已保存至: {curve_path}")
plt.show()


In [ ]:
#@title 6. 打包下载实验结果
import subprocess
from google.colab import files

archive_name = "runs_e10_vector.tar.gz"
subprocess.run(["tar", "czf", archive_name, "runs/vector_board_10x10_seed0"], check=True)
print(f"已打包: {archive_name}")
files.download(archive_name)
